# Exploración: kernels, Kernel PCA y SVM

Notebook de exploración (opcional). Usa el paquete `rkhs_credit` y la configuración por defecto.
Ejecutarlo desde la carpeta `notebooks/` con el paquete instalado (`pip install -e .`).
Requiere además `pip install notebook` (no está en `requirements.txt`).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Las rutas del YAML son relativas a la raíz del repositorio.
if Path.cwd().name == "notebooks":
    os.chdir("..")

from rkhs_credit import plots
from rkhs_credit.config import load_config, with_overrides
from rkhs_credit.data import TabularPreprocessor, load_and_split
from rkhs_credit.evaluation import CostMatrix, classification_metrics
from rkhs_credit.kernels import LinearKernel, PolynomialKernel, RBFKernel
from rkhs_credit.kpca import KernelPCA
from rkhs_credit.numerics import spectral_summary
from rkhs_credit.pipeline import build_pipeline

config = load_config("configs/default.yaml")
plots.apply_style()

## 1. Datos y preprocesamiento (ajustado solo con train)

In [ ]:
X_train, X_test, y_train, y_test = load_and_split(config)
preprocessor = TabularPreprocessor().fit(X_train)
features = preprocessor.transform(X_train)
print(X_train.shape, "->", features.shape, "| tasa de bad en train:", y_train.mean())
X_train.head()

## 2. Kernels componibles y su matriz de Gram

In [ ]:
kernel = 0.5 * LinearKernel() + RBFKernel(gamma=0.01) * PolynomialKernel(degree=2, gamma=0.01)
print(kernel)
K = kernel.gram(features)
summary = spectral_summary(K, jitter=1e-8)
print("simétrica:", np.array_equal(K, K.T))
print(f"autovalor mínimo = {summary.lambda_min:.3e}")
print(f"número de condición = {summary.condition_number:.3e}")

## 3. Kernel PCA: espectro y proyección

In [ ]:
kpca = KernelPCA(RBFKernel(gamma=0.03))
scores = kpca.fit_transform(features)
pca = KernelPCA(LinearKernel())
scores_linear = pca.fit_transform(features)

fig = plots.plot_projection_comparison(
    {
        "Kernel PCA (RBF)": (scores[:, :2], kpca.explained_variance_ratio_[:2]),
        "PCA lineal": (scores_linear[:, :2], pca.explained_variance_ratio_[:2]),
    },
    y_train,
)
fig = plots.plot_spectrum(
    {"Kernel PCA (RBF)": kpca.spectrum_, "PCA lineal": pca.spectrum_}, n_show=250
)
plt.show()

## 4. SVM propia: efecto de `C` y `gamma`

In [ ]:
cost = CostMatrix.from_config(config)
for C, gamma in [(3.0, 0.003), (1.0, 0.01), (10.0, 0.1)]:
    model = build_pipeline(with_overrides(config, {"svm.C": C, "svm.kernel.gamma": gamma}))
    model.fit(X_train, y_train)
    metrics = classification_metrics(
        y_test, model.predict(X_test), model.decision_function(X_test), cost
    )
    print(
        f"C={C:<5} gamma={gamma:<6} vectores de soporte={model.svm.support_.size:3d} "
        f"iteraciones={model.svm.n_iter_:5d}  AUC={metrics['auc']:.3f}  costo={metrics['cost']:.0f}"
    )

## 5. Ideas para seguir

- Cambiar el kernel en `configs/default.yaml` (por ejemplo una suma `linear + rbf`) y volver a correr
  `python -m rkhs_credit.train`.
- Activar `kpca.enabled` y variar `kpca.n_components`.
- Implementar un kernel para variables categóricas (ver «Roadmap» en `docs/report.pdf`).